# ToTTI v1 — fine-tune opus-mt-en-it (78M) on distilled + human-anchor data

Runtime: **T4 GPU**. Expected cost ~1-2 units. Data uploaded: AMI/ICSI-derived distilled
pairs, the Tatoeba anchor (CC BY 2.0 FR), the Europarl slice (statmt: no known copyright
restrictions) and FLORES-200 — the private test set never comes here; accepting scores run
at home against the pre-registered gate in config-v1.yaml (domain >= 54.7, tc-big's
zero-shot). Run all, upload `totti-v1-bundle.zip` when asked, authorize Drive: results are
PARKED ON DRIVE at the end.

In [ ]:
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu_info)
assert gpu_info and "failed" not in gpu_info[0], "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
!pip -q install "transformers<5" sacrebleu ctranslate2 sentencepiece accelerate protobuf

In [ ]:
# Upload totti-v1-bundle.zip (config-v1.yaml, train.py, convert.sh, data/).
from google.colab import files

uploaded = files.upload()
!unzip -o -q totti-v1-bundle.zip -d /content/totti
!wc -l /content/totti/data/*

In [ ]:
# Training with the config echo first (so the log proves what runs), then the
# provisional FLORES verdict. ~10-25 min on a T4.
!cd /content/totti && python train.py --config config-v1.yaml --out out

In [ ]:
# CT2 int8 conversion of the best checkpoint.
!cd /content/totti && bash convert.sh out/best out/opus-mt-en-it-totti-v1-ct2
!du -sh /content/totti/out/opus-mt-en-it-totti-v1-ct2

In [ ]:
# Park BOTH results on Drive first (never trust the tab with gigabytes):
# the CT2 model for the app, and the fp16 best checkpoint for the alpha-sweep at home.
!cd /content/totti/out && zip -r -q /content/totti-v1-ct2.zip opus-mt-en-it-totti-v1-ct2
!cd /content/totti/out && zip -r -q /content/totti-v1-best-fp16.zip best
from google.colab import drive

drive.mount("/content/drive")
!cp /content/totti-v1-ct2.zip /content/totti-v1-best-fp16.zip /content/drive/MyDrive/
!ls -lah /content/drive/MyDrive/totti-v1-*.zip
print("fatto: scaricali da drive.google.com, poi Runtime -> Disconnetti ed elimina runtime")

## At home

Download both zips from Drive, then tell Claude: the alpha-sweep (student x tc-big base) and
the accepting scores against the pre-registered gate run on the Mac. Delete the zips from
Drive afterwards to free the space.